In [ ]:
from typing import Annotated, TypedDict, List, Dict, Any, Optional
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_async_playwright_browser
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from IPython.display import Image, display
import gradio as gr
import uuid
from dotenv import load_dotenv
import os
from math_tools import add, sub, mul, div

In [ ]:
load_dotenv(override=True)

In [ ]:
class State(TypedDict):
    messages: Annotated[List[Any], add_messages]

In [ ]:
tools = [add, sub, mul, div]


worker_llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    #model="anthropic/claude-sonnet-4.6",
    base_url="https://openrouter.ai/api/v1",
    openai_api_key=os.getenv("OPENROUTER_API_KEY")
)


worker_llm_with_tools = worker_llm.bind_tools(tools)

In [ ]:
def worker(state: State) -> Dict[str, Any]:
    system_message = f"""You are a mathematician. You solve math questions in a clever way"""
    
    
    # Add in the system message

    found_system_message = False
    messages = state["messages"]
    for message in messages:
        if isinstance(message, SystemMessage):
            message.content = system_message
            found_system_message = True
    
    if not found_system_message:
        messages = [SystemMessage(content=system_message)] + messages
    
    # Invoke the LLM with tools
    response = worker_llm_with_tools.invoke(messages)
    
    # Return updated state
    return {
        "messages": [response],
    }

In [ ]:
def worker_router(state: State) -> str:
    last_message = state["messages"][-1]
    
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"

In [ ]:
# Set up Graph Builder with State
graph_builder = StateGraph(State)

# Add nodes
graph_builder.add_node("worker", worker)
graph_builder.add_node("tools", ToolNode(tools=tools))

# Add edges
graph_builder.add_conditional_edges("worker", tools_condition, "tools")
graph_builder.add_edge("tools", "worker")
graph_builder.add_edge(START, "worker")

# Compile the graph
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
thread = 1


def process_message(message, history, thread):

    config = {"configurable": {"thread_id": thread}}

    state = {
        "messages": message
    }
    result = graph.invoke(state, config=config)

    return result["messages"][-1].content

In [ ]:

with gr.Blocks(theme=gr.themes.Default(primary_hue="emerald")) as demo:
    gr.Markdown("## Sidekick Personal Co-worker")
    gr.ChatInterface(fn=process_message)
    
demo.launch()